# 02 Entity Extraction With Qwen

Run dictionary/rule-first entity extraction with local Qwen augmentation and resume state.

Partition this notebook by strategy. Keep `SELECTED_SOURCES` in the shared contract for bookkeeping, but do not split entity extraction by source in Kaggle.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

CORPUS_SLUG = 'tuvi-golden-corpus'
SCRIPTS_SLUG = 'tuvi-battu-scripts'

ALL_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child', 'chunk_semantic_embedding_bge_m3']
SOURCE_IDS = ['TVKL', 'TVNL', 'TVHS', 'TVGM']
RUN_TAG = 'part_a'
PARTITION_MODE = 'strategy'
SELECTED_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child']
SELECTED_SOURCES = list(SOURCE_IDS)

CORPUS_DIR = Path(f'/kaggle/input/{CORPUS_SLUG}/benchmark/tuvi_golden_dataset')
SCRIPTS_DIR = Path(f'/kaggle/input/{SCRIPTS_SLUG}')
if not CORPUS_DIR.exists():
    CORPUS_DIR = Path.cwd() / 'benchmark' / 'tuvi_golden_dataset'
if not SCRIPTS_DIR.exists():
    SCRIPTS_DIR = Path.cwd()

ACTIVE_STRATEGIES = [item for item in ALL_STRATEGIES if item in SELECTED_STRATEGIES]
if not ACTIVE_STRATEGIES:
    raise ValueError('No active strategies selected')

OUTPUT_ROOT = Path('/kaggle/working/w3_local_outputs')
REPORTS = OUTPUT_ROOT / 'reports'
STATE = OUTPUT_ROOT / 'state'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
REPORTS.mkdir(parents=True, exist_ok=True)
STATE.mkdir(parents=True, exist_ok=True)

print('CORPUS_DIR =', CORPUS_DIR)
print('SCRIPTS_DIR =', SCRIPTS_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print(json.dumps({'run_tag': RUN_TAG, 'partition_mode': PARTITION_MODE, 'active_strategies': ACTIVE_STRATEGIES, 'selected_sources_recorded_only': SELECTED_SOURCES}, ensure_ascii=False, indent=2))

In [ ]:
for strategy in ACTIVE_STRATEGIES:
    cmd = [
        sys.executable, '-B', str(SCRIPTS_DIR / 'scripts' / 'extract_entities.py'),
        '--input', str(OUTPUT_ROOT / 'chunks' / strategy),
        '--output', str(OUTPUT_ROOT / 'entities' / strategy / f'{strategy}_entities.jsonl'),
        '--chunking-strategy', strategy,
        '--review-output', str(REPORTS / f'{strategy}_entity_review.json'),
        '--partial-summary-output', str(REPORTS / f'{strategy}_entity_summary.json'),
        '--state-output', str(STATE / f'{strategy}_entity_state.json'),
        '--resume',
        '--llm-backend', 'local',
        '--model', 'Qwen/Qwen2.5-7B-Instruct',
        '--local-llm-quantization', '4bit',
        '--local-llm-max-new-tokens', '1024',
    ]
    print('Running strategy =', strategy)
    subprocess.run(cmd, cwd=SCRIPTS_DIR, check=True)

print('Skipped strategies =', [item for item in ALL_STRATEGIES if item not in ACTIVE_STRATEGIES])

In [ ]:
for strategy in ACTIVE_STRATEGIES:
    summary_path = REPORTS / f'{strategy}_entity_summary.json'
    review_path = REPORTS / f'{strategy}_entity_review.json'
    state_path = STATE / f'{strategy}_entity_state.json'
    print(strategy)
    print('  summary =', summary_path.exists(), summary_path)
    print('  review  =', review_path.exists(), review_path)
    print('  state   =', state_path.exists(), state_path)
print('Skipped strategies =', [item for item in ALL_STRATEGIES if item not in ACTIVE_STRATEGIES])